# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import random

import seaborn as sns #For plotting graphs
sns.set_style("whitegrid")

from optbinning import OptimalBinning

import warnings
warnings.filterwarnings("ignore")

(CVXPY) May 25 01:45:09 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.6.2534). Expected < 9.5.0.Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) May 25 01:45:09 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.6.2534). Expected < 9.5.0.Please open a feature request on cvxpy to enable support for this version.')


# Read data

In [2]:
df_train_tx = pd.read_csv('/Users/pradark/01. Work/Toast/data/Lending_default_train_tx.csv')
df_train_acc = pd.read_csv('/Users/pradark/01. Work/Toast/data/Lending_default_train_account.csv')
df_train_label = pd.read_csv('/Users/pradark/01. Work/Toast/data/Lending_default_train_label.csv')

df_holdout_tx = pd.read_csv('/Users/pradark/01. Work/Toast/data/Lending_default_holdout_tx.csv')
df_holdout_acc = pd.read_csv('/Users/pradark/01. Work/Toast/data/Lending_default_holdout_account.csv')

In [3]:
df_train_tx = df_train_tx.loc[:, ~df_train_tx.columns.str.contains('^Unnamed')]
df_train_acc = df_train_acc.loc[:, ~df_train_acc.columns.str.contains('^Unnamed')]
df_train_label = df_train_label.loc[:, ~df_train_label.columns.str.contains('^Unnamed')]

df_holdout_tx = df_holdout_tx.loc[:, ~df_holdout_tx.columns.str.contains('^Unnamed')]
df_holdout_acc = df_holdout_acc.loc[:, ~df_holdout_acc.columns.str.contains('^Unnamed')]

In [4]:
def check_missing(data):
    MissTotal = data.isnull().sum().sort_values(ascending=False)
    percent = round((data.isnull().sum()/list(data.shape)[0]*100),2).sort_values(ascending=False)
    count = data.isnull().count().sort_values(ascending=False)
    missing_data = pd.concat([MissTotal, percent, count], axis=1, keys=['MissingTotal', 'MissingPercent', 'Total'])
    # print(missing_data[missing_data['MissingPercent']>0].head(10))
    print(missing_data.head(10))
    
check_missing(df_train_tx)
check_missing(df_train_acc)
check_missing(df_train_label)

check_missing(df_holdout_tx)
check_missing(df_holdout_acc)

                   MissingTotal  MissingPercent    Total
Restaurant_ID                 0             0.0  3510679
Tx_date                       0             0.0  3510679
processing_volume             0             0.0  3510679
Tx_hours                      0             0.0  3510679
                     MissingTotal  MissingPercent  Total
Ownership_type                220            2.03  10812
Restaurant_catagory             3            0.03  10812
Restaurant_ID                   0            0.00  10812
Market_segment                  0            0.00  10812
               MissingTotal  MissingPercent  Total
Restaurant_ID             0             0.0  10812
loan_default              0             0.0  10812
                   MissingTotal  MissingPercent    Total
Restaurant_ID                 0             0.0  1471016
Tx_date                       0             0.0  1471016
processing_volume             0             0.0  1471016
Tx_hours                      0             0.0  

In [5]:
df_train_tx.nunique()

Restaurant_ID          10812
Tx_date                  365
processing_volume    1029162
Tx_hours                  25
dtype: int64

In [6]:
df_train_tx['Tx_mm'] = df_train_tx['Tx_date'].str[5:7]
df_train_tx['Tx_yyyy'] = df_train_tx['Tx_date'].str[0:4]
df_train_tx['Tx_yyyymm'] = (df_train_tx['Tx_date'].str[0:4]).astype(int)*100 + (df_train_tx['Tx_date'].str[5:7]).astype(int)

In [7]:
from datetime import datetime, timedelta
import datetime
ed_dt = '2022-02-01'
datetime_obj = datetime.datetime.strptime(ed_dt, '%Y-%m-%d')
ed_dt_30days = str(datetime_obj.date() - timedelta(days=30)) # end date minus 30 days
ed_dt_60days = str(datetime_obj.date() - timedelta(days=60)) # end date minus 60 days
ed_dt_90days = str(datetime_obj.date() - timedelta(days=90)) # end date minus 90 days
ed_dt_120days = str(datetime_obj.date() - timedelta(days=120)) # end date minus 120 days
ed_dt_150days = str(datetime_obj.date() - timedelta(days=150)) # end date minus 150 days
ed_dt_180days = str(datetime_obj.date() - timedelta(days=180)) # end date minus 180 days

def f_30days_ago(x):
    if x >= ed_dt_30days: return 1
    else: return 0

df_train_tx['_30days_ago'] = df_train_tx['Tx_date'].apply(f_30days_ago)

def f_90days_ago(x):
    if x >= ed_dt_90days: return 1
    else: return 0

df_train_tx['_90days_ago'] = df_train_tx['Tx_date'].apply(f_90days_ago)

def f_180days_ago(x):
    if x >= ed_dt_180days: return 1
    else: return 0

df_train_tx['_180days_ago'] = df_train_tx['Tx_date'].apply(f_180days_ago)

def f_180days_monthly(x):
    if x >= ed_dt_30days: return '0-30days'
    elif ed_dt_30days >= x >= ed_dt_60days: return '30-60days_ago'
    elif ed_dt_60days >= x >= ed_dt_90days: return '60-90days_ago'
    elif ed_dt_90days >= x >= ed_dt_120days: return '90-120days_ago'
    elif ed_dt_120days >= x >= ed_dt_150days: return '120-150days_ago'
    elif ed_dt_150days >= x >= ed_dt_180days: return '150-180days_ago'
    else: return '180+days_ago'

df_train_tx['monthly'] = df_train_tx['Tx_date'].apply(f_180days_monthly)

In [8]:
temp1 = pd.DataFrame(df_train_tx[df_train_tx['_30days_ago']==1].groupby(['Restaurant_ID','_30days_ago'],as_index=False)['processing_volume'].sum()).rename(columns={'processing_volume':'pro_vol_tot_30days'})
temp1.drop(columns='_30days_ago',inplace=True)
temp2 = pd.DataFrame(df_train_tx[df_train_tx['_30days_ago']==1].groupby(['Restaurant_ID','_30days_ago'],as_index=False)['processing_volume'].mean()).rename(columns={'processing_volume':'pro_vol_avg_30days'})
temp2.drop(columns='_30days_ago',inplace=True)
df_train_flat = pd.merge(temp1,temp2, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_30days_ago']==1].groupby(['Restaurant_ID','_30days_ago'],as_index=False)['processing_volume'].std()).rename(columns={'processing_volume':'pro_vol_std_30days'})
temp.drop(columns='_30days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_90days_ago']==1].groupby(['Restaurant_ID','_90days_ago'],as_index=False)['processing_volume'].sum()).rename(columns={'processing_volume':'pro_vol_tot_90days'})
temp.drop(columns='_90days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_90days_ago']==1].groupby(['Restaurant_ID','_90days_ago'],as_index=False)['processing_volume'].mean()).rename(columns={'processing_volume':'pro_vol_avg_90days'})
temp.drop(columns='_90days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_90days_ago']==1].groupby(['Restaurant_ID','_90days_ago'],as_index=False)['processing_volume'].std()).rename(columns={'processing_volume':'pro_vol_std_90days'})
temp.drop(columns='_90days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_180days_ago']==1].groupby(['Restaurant_ID','_180days_ago'],as_index=False)['processing_volume'].sum()).rename(columns={'processing_volume':'pro_vol_tot_180days'})
temp.drop(columns='_180days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_180days_ago']==1].groupby(['Restaurant_ID','_180days_ago'],as_index=False)['processing_volume'].mean()).rename(columns={'processing_volume':'pro_vol_avg_180days'})
temp.drop(columns='_180days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_180days_ago']==1].groupby(['Restaurant_ID','_180days_ago'],as_index=False)['processing_volume'].std()).rename(columns={'processing_volume':'pro_vol_std_180days'})
temp.drop(columns='_180days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

In [9]:
temp = pd.DataFrame(df_train_tx.groupby(['Restaurant_ID','monthly'],as_index=False)['processing_volume'].mean()).rename(columns={'processing_volume':'pro_vol_avg_monthly'})
pivot1 = temp.pivot(*temp).rename_axis(index=None, columns=None)

pivot1['pro_vol_6mth_chg_pct'] = ((pivot1['0-30days'] - pivot1['150-180days_ago'])/pivot1['0-30days'])*100
pivot1['pro_vol_3mth_chg_pct'] = ((pivot1['0-30days'] - pivot1['60-90days_ago'])/pivot1['0-30days'])*100
pivot1['pro_vol_3mth_max'] = pivot1[['0-30days','30-60days_ago','60-90days_ago']].max(axis=1)
pivot1['pro_vol_3mth_min'] = pivot1[['0-30days','30-60days_ago','60-90days_ago',]].min(axis=1)
pivot1['pro_vol_3mth_max_min'] = pivot1['pro_vol_3mth_max'] - pivot1['pro_vol_3mth_min']
pivot1['pro_vol_6mth_max'] = pivot1[['0-30days','30-60days_ago','60-90days_ago','90-120days_ago',\
                                     '120-150days_ago','150-180days_ago']].max(axis=1)
pivot1['pro_vol_6mth_min'] = pivot1[['0-30days','30-60days_ago','60-90days_ago','90-120days_ago',\
                                     '120-150days_ago','150-180days_ago']].min(axis=1)
pivot1['pro_vol_6mth_max_min'] = pivot1['pro_vol_6mth_max'] - pivot1['pro_vol_6mth_min']

add_list = ['pro_vol_6mth_chg_pct','pro_vol_3mth_chg_pct','pro_vol_3mth_max','pro_vol_3mth_min','pro_vol_3mth_max_min',\
           'pro_vol_6mth_max','pro_vol_6mth_min','pro_vol_6mth_max_min']
df_train_flat = pd.merge(df_train_flat,pivot1[add_list], left_on=['Restaurant_ID'],right_index=True,how = 'inner')

In [10]:
temp = pd.DataFrame(df_train_tx[df_train_tx['_30days_ago']==1].groupby(['Restaurant_ID','_30days_ago'],as_index=False)['Tx_hours'].sum()).rename(columns={'Tx_hours':'hours_tot_30days'})
temp.drop(columns='_30days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_30days_ago']==1].groupby(['Restaurant_ID','_30days_ago'],as_index=False)['Tx_hours'].mean()).rename(columns={'Tx_hours':'hours_avg_30days'})
temp.drop(columns='_30days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_30days_ago']==1].groupby(['Restaurant_ID','_30days_ago'],as_index=False)['Tx_hours'].std()).rename(columns={'Tx_hours':'hours_std_30days'})
temp.drop(columns='_30days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_90days_ago']==1].groupby(['Restaurant_ID','_90days_ago'],as_index=False)['Tx_hours'].sum()).rename(columns={'Tx_hours':'hours_tot_90days'})
temp.drop(columns='_90days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_90days_ago']==1].groupby(['Restaurant_ID','_90days_ago'],as_index=False)['Tx_hours'].mean()).rename(columns={'Tx_hours':'hours_avg_90days'})
temp.drop(columns='_90days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_90days_ago']==1].groupby(['Restaurant_ID','_90days_ago'],as_index=False)['Tx_hours'].std()).rename(columns={'Tx_hours':'hours_std_90days'})
temp.drop(columns='_90days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_180days_ago']==1].groupby(['Restaurant_ID','_180days_ago'],as_index=False)['Tx_hours'].sum()).rename(columns={'Tx_hours':'hours_tot_180days'})
temp.drop(columns='_180days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_180days_ago']==1].groupby(['Restaurant_ID','_180days_ago'],as_index=False)['Tx_hours'].mean()).rename(columns={'Tx_hours':'hours_avg_180days'})
temp.drop(columns='_180days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(df_train_tx[df_train_tx['_180days_ago']==1].groupby(['Restaurant_ID','_180days_ago'],as_index=False)['Tx_hours'].std()).rename(columns={'Tx_hours':'hours_std_180days'})
temp.drop(columns='_180days_ago',inplace=True)
df_train_flat = pd.merge(df_train_flat,temp, on=['Restaurant_ID'],how = 'inner')

In [11]:
temp = pd.DataFrame(df_train_tx.groupby(['Restaurant_ID','monthly'],as_index=False)['Tx_hours'].mean()).rename(columns={'Tx_hours':'hours_avg_monthly'})
pivot1 = temp.pivot(*temp).rename_axis(index=None, columns=None)

pivot1['hours_6mth_chg_pct'] = ((pivot1['0-30days'] - pivot1['150-180days_ago'])/pivot1['0-30days'])*100
pivot1['hours_3mth_chg_pct'] = ((pivot1['0-30days'] - pivot1['60-90days_ago'])/pivot1['0-30days'])*100

pivot1['hours_3mth_max'] = pivot1[['0-30days','30-60days_ago','60-90days_ago']].max(axis=1)
pivot1['hours_3mth_min'] = pivot1[['0-30days','30-60days_ago','60-90days_ago',]].min(axis=1)
pivot1['hours_3mth_max_min'] = pivot1['hours_3mth_max'] - pivot1['hours_3mth_min']
pivot1['hours_6mth_max'] = pivot1[['0-30days','30-60days_ago','60-90days_ago','90-120days_ago',\
                                     '120-150days_ago','150-180days_ago']].max(axis=1)
pivot1['hours_6mth_min'] = pivot1[['0-30days','30-60days_ago','60-90days_ago','90-120days_ago',\
                                     '120-150days_ago','150-180days_ago']].min(axis=1)
pivot1['hours_6mth_max_min'] = pivot1['hours_6mth_max'] - pivot1['hours_6mth_min']

add_list = ['hours_6mth_chg_pct','hours_3mth_chg_pct','hours_3mth_max','hours_3mth_min','hours_3mth_max_min',\
           'hours_6mth_max','hours_6mth_min','hours_6mth_max_min']

df_train_flat = pd.merge(df_train_flat,pivot1[add_list], left_on=['Restaurant_ID'],right_index=True,how = 'inner')

In [12]:
df_train_flat.head()

,Restaurant_ID,pro_vol_tot_30days,pro_vol_avg_30days,pro_vol_std_30days,pro_vol_tot_90days,pro_vol_avg_90days,pro_vol_std_90days,pro_vol_tot_180days,pro_vol_avg_180days,pro_vol_std_180days,...,hours_avg_180days,hours_std_180days,hours_6mth_chg_pct,hours_3mth_chg_pct,hours_3mth_max,hours_3mth_min,hours_3mth_max_min,hours_6mth_max,hours_6mth_min,hours_6mth_max_min
0,0000ed50-b0bb-48ab-80d5-7237f8ab88bd,120065.12,3873.068387,2205.346279,450457.85,4950.086264,2244.608380,991267.74,5476.617348,2054.833616,...,10.176796,2.000476,-5.027322,0.054645,10.533333,9.833333,0.700000,10.533333,9.833333,0.700000
1,0004ee6d-7488-47fb-bc35-188e5cac9b32,20659.50,666.435484,508.112774,50422.69,554.095495,504.010427,107478.60,593.804420,480.767841,...,5.132597,3.860640,-10.848485,0.424242,5.322581,3.266667,2.055914,5.966667,3.266667,2.700000
2,0008c563-adbd-473c-80d3-b94f0c19cc8a,54481.41,1757.464839,779.471888,183793.52,2019.709011,1054.212645,403623.85,2229.966022,1078.803758,...,13.585635,2.123836,-1.874510,3.474510,13.709677,12.933333,0.776344,13.966667,12.933333,1.033333
3,00181386-d5b9-4551-9286-1dafed67a8e6,506015.39,16323.077097,7334.346977,1518878.73,16690.975055,8296.262360,2664274.67,14719.749558,8179.638462,...,11.364641,1.991557,0.408542,1.847725,11.580645,10.166667,1.413978,11.766667,10.166667,1.600000
4,00332a25-a236-4174-95a8-6b3ce8cd5def,81079.78,2615.476774,1325.588923,304199.33,3342.849780,2168.617066,553155.06,3056.105304,1875.599354,...,8.397790,1.812547,17.983683,2.447552,9.533333,9.000000,0.533333,9.533333,7.066667,2.466667


In [13]:
data = df_train_tx
data_flat = df_train_flat

In [14]:
std_list = [x for x in data_flat.filter(regex='std')]
std_list

['pro_vol_std_30days',
 'pro_vol_std_90days',
 'pro_vol_std_180days',
 'hours_std_30days',
 'hours_std_90days',
 'hours_std_180days']

In [15]:
keep_list = ['Restaurant_ID'] + std_list 
keep_list

['Restaurant_ID',
 'pro_vol_std_30days',
 'pro_vol_std_90days',
 'pro_vol_std_180days',
 'hours_std_30days',
 'hours_std_90days',
 'hours_std_180days']

In [16]:
data = pd.merge(data,data_flat[keep_list], on=['Restaurant_ID'],how = 'inner')

In [17]:
data.head()

,Restaurant_ID,Tx_date,processing_volume,Tx_hours,Tx_mm,Tx_yyyy,Tx_yyyymm,_30days_ago,_90days_ago,_180days_ago,monthly,pro_vol_std_30days,pro_vol_std_90days,pro_vol_std_180days,hours_std_30days,hours_std_90days,hours_std_180days
0,cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5,2021-02-02,3778.49,12.0,02,2021,202102,0,0,0,180+days_ago,1518.254802,2589.631849,2274.791801,1.722089,1.978023,1.562922
1,cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5,2021-02-03,4456.12,14.0,02,2021,202102,0,0,0,180+days_ago,1518.254802,2589.631849,2274.791801,1.722089,1.978023,1.562922
2,cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5,2021-02-04,4868.75,12.0,02,2021,202102,0,0,0,180+days_ago,1518.254802,2589.631849,2274.791801,1.722089,1.978023,1.562922
3,cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5,2021-02-05,5168.63,14.0,02,2021,202102,0,0,0,180+days_ago,1518.254802,2589.631849,2274.791801,1.722089,1.978023,1.562922
4,cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5,2021-02-06,6583.37,14.0,02,2021,202102,0,0,0,180+days_ago,1518.254802,2589.631849,2274.791801,1.722089,1.978023,1.562922


In [18]:
def pro_vol_lt_std_30days(df):
    if df['processing_volume'] < df['pro_vol_std_30days'] and df['_30days_ago'] == 1: return 1
    else: return 0

data['pro_vol_lt_30days_std_count'] = data.apply(pro_vol_lt_std_30days,axis=1)

def pro_vol_lt_std_90days(df):
    if df['processing_volume'] < df['pro_vol_std_90days'] and df['_90days_ago'] == 1: return 1
    else: return 0

data['pro_vol_lt_90days_std_count'] = data.apply(pro_vol_lt_std_90days,axis=1)

def pro_vol_lt_std_180days(df):
    if df['processing_volume'] < df['pro_vol_std_180days'] and df['_180days_ago'] == 1: return 1
    else: return 0

data['pro_vol_lt_180days_std_count'] = data.apply(pro_vol_lt_std_180days,axis=1)

def hours_lt_std_30days(df):
    if df['Tx_hours'] < df['hours_std_30days'] and df['_30days_ago'] == 1: return 1
    else: return 0

data['hours_lt_30days_std_count'] = data.apply(hours_lt_std_30days,axis=1)

def hours_lt_std_90days(df):
    if df['Tx_hours'] < df['hours_std_90days'] and df['_90days_ago'] == 1: return 1
    else: return 0

data['hours_lt_90days_std_count'] = data.apply(hours_lt_std_90days,axis=1)

def hours_lt_std_180days(df):
    if df['Tx_hours'] < df['hours_std_180days'] and df['_180days_ago'] == 1: return 1
    else: return 0

data['hours_lt_180days_std_count'] = data.apply(hours_lt_std_180days,axis=1)

In [19]:
temp = pd.DataFrame(data[data['_30days_ago']==1].groupby(['Restaurant_ID','_30days_ago'],as_index=False)['pro_vol_lt_30days_std_count'].sum())
temp.drop(columns='_30days_ago',inplace=True)
data_flat = pd.merge(data_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(data[data['_90days_ago']==1].groupby(['Restaurant_ID','_90days_ago'],as_index=False)['pro_vol_lt_90days_std_count'].sum())
temp.drop(columns='_90days_ago',inplace=True)
data_flat = pd.merge(data_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(data[data['_180days_ago']==1].groupby(['Restaurant_ID','_180days_ago'],as_index=False)['pro_vol_lt_180days_std_count'].sum())
temp.drop(columns='_180days_ago',inplace=True)
data_flat = pd.merge(data_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(data[data['_30days_ago']==1].groupby(['Restaurant_ID','_30days_ago'],as_index=False)['hours_lt_30days_std_count'].sum())
temp.drop(columns='_30days_ago',inplace=True)
data_flat = pd.merge(data_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(data[data['_90days_ago']==1].groupby(['Restaurant_ID','_90days_ago'],as_index=False)['hours_lt_90days_std_count'].sum())
temp.drop(columns='_90days_ago',inplace=True)
data_flat = pd.merge(data_flat,temp, on=['Restaurant_ID'],how = 'inner')

temp = pd.DataFrame(data[data['_180days_ago']==1].groupby(['Restaurant_ID','_180days_ago'],as_index=False)['hours_lt_180days_std_count'].sum())
temp.drop(columns='_180days_ago',inplace=True)
data_flat = pd.merge(data_flat,temp, on=['Restaurant_ID'],how = 'inner')

In [20]:
temp

,Restaurant_ID,hours_lt_180days_std_count
0,0000ed50-b0bb-48ab-80d5-7237f8ab88bd,5
1,0004ee6d-7488-47fb-bc35-188e5cac9b32,65
2,0008c563-adbd-473c-80d3-b94f0c19cc8a,3
3,00181386-d5b9-4551-9286-1dafed67a8e6,4
4,00332a25-a236-4174-95a8-6b3ce8cd5def,0
...,...,...
10807,ffe71fea-e49b-4991-934e-14a8a696eb9f,1
10808,ffeb161e-1b7e-40f0-ac5f-7f9eca1030ba,0
10809,fff537e1-a6f5-4ccf-80bf-6f4465144694,1
10810,fff78879-a813-4870-acb2-7e9343a5fbc1,6
